# WCF sensitivity shard 29

Shard **29** of 48 for the frozen sensitivity study (`WCF-SENSITIVITY-v1`, manifest checksum `1a779680cca9e037...`).

This shard runs **8 cells** over blocks `align`, `income_onefactor`, `propensity`, `sym` with methods `cwdb_dr`, `cwdb_dr_flex`, `cwdb_dr_oracle`.

Estimated reference cost is about **69 minutes** single-threaded (from results/wcf_sensitivity/cost_pilot.json, linear in K and M). The estimate is rough and only balances shards; Colab cores are slower, so allow two to three times that.

Every notebook is self-contained: it unpacks an embedded source archive, embeds its own manifest slice, checkpoints the parquet after every cell, and ends by downloading one zip. The local tmux run is the primary path; this shard is the contingency copy.


In [ ]:
# Thread pinning MUST happen before NumPy or SciPy are imported. OpenMP sizes
# its pool at initialisation, so setting these afterwards is silently
# ineffective.
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS', 'R_NUM_THREADS'):
    os.environ[_v] = '1'
print('threads pinned to 1')

## 1. Embedded source

The working tree contains uncommitted sensitivity code, so no pinned clone can reproduce it. This cell decodes a base64 ZIP of `src/wasserstein_causal_forests/` (plus the R drivers when this shard contains forest methods), extracts it to a temporary directory, and puts the extracted `src` on `sys.path`.


In [ ]:
import base64
import pathlib
import sys
import tempfile
import zipfile

SOURCE_ARCHIVE_SHA256 = '484439b617535daad0c2f90a8a54dfc41bca3e170aa7138d51038b35c742796f'
SOURCE_ARCHIVE_B64 = '''\
...
ADcAdBQAAIsTAwAAAA==
'''

workdir = pathlib.Path(tempfile.mkdtemp(prefix='wcf_sensitivity_'))
archive_path = workdir / 'wcf_source.zip'
archive_path.write_bytes(base64.b64decode(SOURCE_ARCHIVE_B64))
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(workdir)
source_directory = workdir / 'src'
sys.path.insert(0, str(source_directory))
import wasserstein_causal_forests
print('embedded source:', wasserstein_causal_forests.__file__)
print('source archive sha256:', SOURCE_ARCHIVE_SHA256)
print('manifest checksum: 1a779680cca9e037ec3f70be86a98943a59ff37489763d24841c0a9c59d48805')


## 3. Registration and this shard's manifest slice

The frozen registry is applied after the sensitivity registration, so the exact frozen estimator settings win over the static defaults.


In [ ]:
import json
from wasserstein_causal_forests.g3 import runner
from wasserstein_causal_forests.g3.sensitivity_dgps import (
    register_sensitivity_dgps,
)
from wasserstein_causal_forests.g3.sensitivity_methods import (
    register_sensitivity_methods,
)
from wasserstein_causal_forests.g3.wcf_sensitivity import (
    apply_method_registry,
)

SHARD_INDEX = 29
SHARD_TOTAL = 48
ESTIMATED_SECONDS = 4159.82
MANIFEST_SLICE = json.loads('''{"contract_id": "WCF-SENSITIVITY-v1", "manifest_checksum": "1a779680cca9e037ec3f70be86a98943a59ff37489763d24841c0a9c59d48805", "estimator_source_hash": "c105c0aaff1b0b8e598799351cfb490845ef2253adffd909364c0670e0505d92", "blocks": ["income_onefactor", "income_baselines", "sym", "align", "propensity"], "cell_blocks": {"4520409a687bcd3f": ["align"], "8b6cf3c9774e7e98": ["align"], "914b3bc1c603cfbe": ["propensity"], "73a0e615847dfd8d": ["propensity"], "1eff94407286d98d": ["income_onefactor"], "a61204848e4ea2ba": ["income_onefactor"], "622171da4a749a54": ["sym"], "5dfd39609dd3e986": ["sym"]}, "method_registry": {"cwdb_dr": {"role": "variant", "adapter": "cwdb_dr", "produces_law": true, "cross_fitted": true, "parameters": {"contrast_candidates": [0.0, 50.0, 500.0], "n_folds": 3, "common_grid_levels": "COMMON199+INTERIOR", "propensity_factory": "logistic"}, "propensity_clip": [0.02, 0.98], "boosting_budget": {"n_estimators": 100, "learning_rate": 0.12, "max_depth": 4, "min_samples_leaf": 10, "min_arm_leaf": 5, "collision_epsilon": 0.001}, "estimator_source_hash": "c105c0aaff1b0b8e598799351cfb490845ef2253adffd909364c0670e0505d92"}, "cwdb_dr_flex": {"role": "variant", "adapter": "cwdb_dr", "produces_law": true, "cross_fitted": true, "parameters": {"contrast_candidates": [0.0, 50.0, 500.0], "n_folds": 3, "common_grid_levels": "COMMON199+INTERIOR", "propensity_factory": "hist_gradient_boosting"}, "propensity_clip": [0.02, 0.98], "boosting_budget": {"n_estimators": 100, "learning_rate": 0.12, "max_depth": 4, "min_samples_leaf": 10, "min_arm_leaf": 5, "collision_epsilon": 0.001}, "estimator_source_hash": "c105c0aaff1b0b8e598799351cfb490845ef2253adffd909364c0670e0505d92"}, "cwdb_dr_oracle": {"role": "diagnostic", "adapter": "cwdb_dr", "produces_law": true, "cross_fitted": true, "parameters": {"contrast_candidates": [0.0, 50.0, 500.0], "n_folds": 3, "common_grid_levels": "COMMON199+INTERIOR", "oracle_propensity": true}, "propensity_clip": [0.02, 0.98], "boosting_budget": {"n_estimators": 100, "learning_rate": 0.12, "max_depth": 4, "min_samples_leaf": 10, "min_arm_leaf": 5, "collision_epsilon": 0.001}, "estimator_source_hash": "c105c0aaff1b0b8e598799351cfb490845ef2253adffd909364c0670e0505d92"}}, "cells": [{"grid": "wcf_sensitivity_v1_align", "dgp": "SYM-IRREL", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 3, "cell_key": "4520409a687bcd3f", "test_seed": 900003}, {"grid": "wcf_sensitivity_v1_align", "dgp": "SYM-IRREL", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 3, "cell_key": "8b6cf3c9774e7e98", "test_seed": 900003}, {"grid": "wcf_sensitivity_v1_flex", "dgp": "SYM-NL", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr_flex", "seed": 5, "cell_key": "914b3bc1c603cfbe", "test_seed": 900005}, {"grid": "wcf_sensitivity_v1_flex", "dgp": "SYM-NL", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr_oracle", "seed": 5, "cell_key": "73a0e615847dfd8d", "test_seed": 900005}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 3, "cell_key": "1eff94407286d98d", "test_seed": 900003}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 25, "method": "cwdb_dr", "seed": 9, "cell_key": "a61204848e4ea2ba", "test_seed": 900009}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-RANDOM", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 9, "cell_key": "622171da4a749a54", "test_seed": 900009}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-RANDOM", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 5, "cell_key": "5dfd39609dd3e986", "test_seed": 900005}]}''')
CELLS = MANIFEST_SLICE['cells']
METHOD_ORDER = ['cwdb_dr', 'cwdb_dr_flex', 'cwdb_dr_oracle']
CONTRACT_ID = MANIFEST_SLICE['contract_id']

register_sensitivity_dgps()
register_sensitivity_methods()
apply_method_registry({'method_registry': MANIFEST_SLICE['method_registry']})
print('shard', SHARD_INDEX, 'of', SHARD_TOTAL, '|', len(CELLS), 'cells')
print('registered methods:', sorted(MANIFEST_SLICE['method_registry']))

## 4. Run

Cells are ordered so one replication's methods stay adjacent and share the cached dense oracle truth. Re-running this cell resumes from the parquet checkpoint; a failed cell is retried rather than treated as done.


In [ ]:
# The runner is a per-cell checkpoint loop: after every cell the accumulated
# rows are rewritten to a temporary parquet and atomically renamed, so an
# interrupted session resumes from the parquet rather than from the start.
import json
import os
import time
from pathlib import Path

import pyarrow  # noqa: F401  (runner.write_rows falls back to JSONL without it)
import pyarrow.parquet as pq
from wasserstein_causal_forests.g3 import common_grid, runner
from wasserstein_causal_forests.g3.manifest import Cell

OUTPUT_DIRECTORY = Path('shard_output')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
PARQUET_PATH = OUTPUT_DIRECTORY / 'wcf_sensitivity_parquet.parquet'
ROWS_PATH = OUTPUT_DIRECTORY / 'rows.jsonl'
LOG_PATH = OUTPUT_DIRECTORY / 'execution_log.jsonl'
FAILURE_PATH = OUTPUT_DIRECTORY / 'failure_rows.jsonl'
CACHE_DIRECTORY = OUTPUT_DIRECTORY / 'cache'
CACHE_DIRECTORY.mkdir(parents=True, exist_ok=True)


def _read_rows(path):
    if not path.exists():
        return []
    return pq.read_table(path).to_pandas().to_dict('records')


def _replication_key(item):
    return (item['grid'], item['dgp'], item['n_train'],
            item['n_grid'], item['n_particles'], item['seed'])


rows = _read_rows(PARQUET_PATH)
by_cell = {}
for row in rows:
    by_cell.setdefault(row['cell_key'], []).append(row)
successful = {key for key, group in by_cell.items()
              if not any(row.get('metric') == 'cell_failure' for row in group)}
failed = {key for key, group in by_cell.items()
          if any(row.get('metric') == 'cell_failure' for row in group)}
if failed:
    # A failed cell is not a resume marker: drop its old rows and retry it.
    rows = [row for row in rows if row['cell_key'] not in failed]
    print(f'resuming: {len(successful)} successful cells, '
          f'retrying {len(failed)} failed cells')

method_order = {name: position for position, name in enumerate(METHOD_ORDER)}
ordered_cells = sorted(
    CELLS,
    key=lambda item: (_replication_key(item), method_order[item['method']]),
)

n_ok = 0
n_failed = 0
n_skipped = 0
started = time.time()
try:
    for position, item in enumerate(ordered_cells, start=1):
        if item['cell_key'] in successful:
            n_skipped += 1
            continue
        cell = Cell(**{key: value for key, value in item.items()
                       if key not in ('cell_key', 'test_seed')})
        assert cell.key == item['cell_key'], 'cell key mismatch in the slice'
        cell_rows = runner.run_cell(
            cell,
            cache_directory=CACHE_DIRECTORY,
            manifest_contract_id=CONTRACT_ID,
            extra_evaluators=(common_grid.evaluate_common_grid,),
        )
        rows.extend(cell_rows)
        status = 'failed' if cell_rows[0].get('status') == 'failed' else 'ok'
        n_ok += status == 'ok'
        n_failed += status == 'failed'
        temporary = PARQUET_PATH.with_suffix('.tmp')
        runner.write_rows(rows, temporary)
        os.replace(temporary, PARQUET_PATH)
        wall_seconds = float(cell_rows[0].get('wall_seconds', 0.0))
        with ROWS_PATH.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(
                {'cell_key': cell.key, 'rows': cell_rows}, default=str) + '\n')
        with LOG_PATH.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps({
                'cell_key': cell.key, 'grid': cell.grid, 'dgp': cell.dgp,
                'n_train': cell.n_train, 'n_grid': cell.n_grid,
                'n_particles': cell.n_particles, 'method': cell.method,
                'seed': cell.seed, 'status': status, 'n_rows': len(cell_rows),
                'wall_seconds': round(wall_seconds, 3),
                'finished_at': time.time(),
            }) + '\n')
        if status == 'failed':
            with FAILURE_PATH.open('a', encoding='utf-8') as handle:
                handle.write(json.dumps(
                    {'cell_key': cell.key, 'rows': cell_rows},
                    default=str) + '\n')
        print(f'[{position}/{len(ordered_cells)}] {cell.dgp} '
              f'n={cell.n_train} K={cell.n_grid} M={cell.n_particles} '
              f'{cell.method} seed={cell.seed}: {status} '
              f'{wall_seconds:.1f}s', flush=True)
except KeyboardInterrupt:
    print('interrupted; re-run this cell to resume from the parquet '
          'checkpoint', flush=True)

print(f'shard finished: {n_ok} ok, {n_failed} failed, {n_skipped} already '
      f'present, {len(rows)} rows, {(time.time() - started) / 60.0:.1f} min')

## 5. Diagnostics

Writes the shard configuration, the embedded manifest slice, dependency versions, and the embedded source hash into `shard_output/` so the bundle is self-describing.


In [ ]:
# Diagnostics for the shard bundle: exact code identity, shard slice, and the
# dependency versions that Colab actually provided.
import json
import platform
import sys
import time
from pathlib import Path

versions = {'python': sys.version.split()[0], 'platform': platform.platform()}
for _name in ('numpy', 'scipy', 'sklearn', 'pandas', 'pyarrow'):
    try:
        _module = __import__(_name)
        versions[_name] = getattr(_module, '__version__', 'unknown')
    except ImportError:
        versions[_name] = None

config = {
    'shard_index': SHARD_INDEX,
    'shard_total': SHARD_TOTAL,
    'contract_id': CONTRACT_ID,
    'manifest_checksum': MANIFEST_SLICE['manifest_checksum'],
    'estimator_source_hash': MANIFEST_SLICE.get('estimator_source_hash'),
    'source_archive_sha256': SOURCE_ARCHIVE_SHA256,
    'blocks': MANIFEST_SLICE['blocks'],
    'methods': METHOD_ORDER,
    'n_cells': len(CELLS),
    'cell_keys': [item['cell_key'] for item in CELLS],
    'estimated_seconds_reference': ESTIMATED_SECONDS,
    'versions': versions,
}
Path('shard_output').mkdir(parents=True, exist_ok=True)
Path('shard_output/shard_config.json').write_text(
    json.dumps(config, indent=2), encoding='utf-8')
Path('shard_output/manifest_slice.json').write_text(
    json.dumps(MANIFEST_SLICE, indent=2), encoding='utf-8')
# The sidecar uses the keys the local merge verifies, so an overflow shard can
# be renamed into results/wcf_sensitivity/shards/ without inventing metadata.
Path('shard_output/wcf_sensitivity_parquet.meta.json').write_text(
    json.dumps({
        'manifest_checksum': MANIFEST_SLICE['manifest_checksum'],
        'estimator_source_hash': MANIFEST_SLICE.get('estimator_source_hash'),
        'contract_id': CONTRACT_ID,
        'updated_at': time.time(),
    }, indent=2), encoding='utf-8')
print(json.dumps(config, indent=2))

## 6. Download the results

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
output_file = "wcf_sensitivity_shard.zip"
with ZipFile(output_file, "w", ZIP_DEFLATED) as archive:
    for path in Path("shard_output").rglob("*"):
        if path.is_file(): archive.write(path, arcname=path.relative_to("shard_output"))
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)

## What to send back

Download the single `wcf_sensitivity_shard.zip`. It contains this shard's
`wcf_sensitivity_parquet.parquet`, its execution log, its failure rows (if
any), the embedded manifest slice, and `shard_config.json` with the embedded
source hash and dependency versions. Do not hand-edit the parquet, the log, or
the failure rows: the merge reconciles every cell key against the frozen
manifest, keeps failures explicit, and refuses duplicates. The local tmux run
remains the primary path; this notebook is the contingency copy.

To feed an overflow bundle into the local merge, unzip it into
`results/wcf_sensitivity/shards/`, rename the parquet to
`shard_colab_<index>.parquet`, and rename `wcf_sensitivity_parquet.meta.json` to
`shard_colab_<index>.meta.json` (for example `shard_colab_00.parquet`); the
sidecar already carries the contract, checksum, and estimator hash the merge
verifies. The `colab_` infix keeps overflow shards from overwriting the local
runner's numeric `shard_000.parquet` files.